# Public historical notebook
Outputs, authentication metadata and personal artifact links have been removed for publication.
This is a historical code record, NOT a complete retraining kit. Do not Run All.
See the stage README and aggregate training history. Private datasets and model archives are not included.


# Colab arxiv nusxasi

Bu VS Code’da ko‘rsatish uchun saqlangan tarixiy notebook. Asl bajarilishning matnli chiqishlari saqlandi. Maxfiy tokenlar yashiriladi, HTML/rasm chiqishlari olinmaydi. Treningni laptopda Run All qilmang. Haqiqiy yangi trening uchun Colab va alohida run kerak.


# ByT5 — 5 000 juftlik bilan full fine-tuning

**Parent: real-392 → yangi 5k bosqich. L4 GPU, 1 epoch, 157 qadam.**
Bu yangi alohida notebook: eski notebook/run/model fayllarini almashtirmang.
Dataset Silver; inson tasdiqlagan Gold emas. 60 development; final test ishlatilmaydi.
Google Drive’da taxminan 30 GiB bo‘sh joy va Colab GPU compute kerak. API/W&B kaliti yo‘q.
Bulutdagi ~2.2 GiB parent model to‘g‘ridan-to‘g‘ri Colab’ga yuklanadi.

## 1. Kichik paketni yuklash va kutubxonalarni tayyorlash
Runtime → Change runtime type → **L4 GPU**. Hamma eski treninglar to‘xtagan bo‘lsin.
Katakni bajaring va `uznorm-quality5k-colab-v1.zip` faylini tanlang. Drive mount kerak emas.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, stat, subprocess, sys, tempfile, uuid, zipfile
from google.colab import files
sys.dont_write_bytecode = True

ZIP = Path('/content/uznorm-quality5k-colab-v1.zip')
EXPECTED_SHA256 = 'bbdced442894ed647057b0999754aa4c94f8175c15989ec49641d072288e8794'
if not ZIP.is_file():
    files.upload()
if not ZIP.is_file() or hashlib.sha256(ZIP.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError('Aynan berilgan ZIP kerak. SHA-256 mos emas; trening boshlanmadi.')
KIT = Path(tempfile.mkdtemp(prefix='quality5k-kit-', dir='/content'))
with zipfile.ZipFile(ZIP) as bundle:
    infos = bundle.infolist()
    if len(infos) > 200 or len({i.filename for i in infos}) != len(infos) or sum(i.file_size for i in infos) > 30*1024**2:
        raise RuntimeError('ZIP tarkibi/hajmi noto‘g‘ri.')
    for info in infos:
        name, part = info.filename, Path(info.filename)
        if part.is_absolute() or '..' in part.parts or ':' in name or chr(92) in name or info.is_dir() or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError('Xavfli ZIP yo‘li.')
        target = KIT / part
        target.parent.mkdir(parents=True, exist_ok=True)
        with bundle.open(info) as incoming, target.open('xb') as outgoing:
            shutil.copyfileobj(incoming, outgoing)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(KIT)+os.pathsep+str(KIT/'src'), PYTHONDONTWRITEBYTECODE='1',
           PYTHONUNBUFFERED='1', USE_TF='0', USE_FLAX='0', WANDB_MODE='disabled')
os.environ.update(USE_TF='0', USE_FLAX='0', WANDB_MODE='disabled')
subprocess.run([sys.executable, '-B', '-u', '-m', 'stagequality.colab_runner', '--package', str(KIT)], env=ENV, check=True)
print('Kutubxonalar tayyorlanmoqda; model yuklanmayapti...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(KIT/'requirements-colab.txt')], check=True)
spec = importlib.util.spec_from_file_location('quality_visible_log', KIT/'notebook_support.py')
support = importlib.util.module_from_spec(spec)
spec.loader.exec_module(support)
print('PAKET TAYYOR. Keyingi katak: Google hisobini tekshirish.')


## 2. Google hisobi va bulutdagi model — faqat tekshiruv
Oldingi real-392 backupi turgan **o‘sha Google hisob** bilan kiring.
Bu katak modelni o‘qitmaydi, vaznlarini yuklamaydi va Drive’ga yozmaydi.


In [ ]:
from google.colab import auth
auth.authenticate_user()
CHECK_LOG = Path('/content') / ('quality-check-' + uuid.uuid4().hex[:12] + '.log')
support.run_visible([sys.executable, '-B', '-u', '-m', 'stagequality.colab_runner',
                     '--package', str(KIT), '--check-cloud'], ENV, CHECK_LOG)


## 3. Full fine-tuningni boshlash / uzilgandan keyin davom ettirish
**`START_TRAINING = True` qilib, aynan shu katakni bajaring.**
Bu barcha eski treninglar to‘xtagani, Silver data, GPU sarfi va yangi Google backup papkasiga tasdiq.
Ikki Colab sessiyasida parallel boshlamang. Eski notebookni `Run all` qilmang.

Jarayon: parent download → SHA-256 → GPU smoke → 60 baseline → training /157 → 60 after → final backup.
Har 25 qadamda backup; u tasdiqlanmaguncha keyingi qadamga o‘tilmaydi. Katta fayl yuklanganda step turishi normal.
Har 10 qadamda progress va taxminiy vaqt, log jim bo‘lsa har 30 soniyada faol jarayon xabari chiqadi.
**Colab uzilishi mumkin**. Aynan shu ZIP bilan qayta bajarsangiz, oxirgi tasdiqlangan 5k checkpoint tiklanadi.
Backupdan keyingi ko‘pi bilan 25 qadam qayta bajarilishi mumkin; eski parentga yashirin qaytish yo‘q.


In [ ]:
START_TRAINING = True
if START_TRAINING:
    LOG = Path('/content') / ('quality-training-' + uuid.uuid4().hex[:12] + '.log')
    command = [sys.executable, '-B', '-u', '-m', 'stagequality.colab_runner', '--package', str(KIT),
               '--train', '--allow-cloud', '--old-stopped', '--allow-silver']
    support.run_visible(command, ENV, LOG)
else:
    print('Boshlash uchun START_TRAINING=True qiling va shu katakni bajaring.')


## 4. Yakuniy natijani ko‘rish
**`QUALITY_STAGE_COMPLETE_CLOUD_VERIFIED step=157 epoch=1`** yoki `QUALITY_ALREADY_COMPLETE`
logidan so‘ng bajaring. `exit code: 0` yolg‘iz o‘zi yetarli emas. Natija eski modelni avtomatik almashtirmaydi.


In [ ]:
if 'LOG' not in globals() or not LOG.is_file():
    raise RuntimeError('Avval 3-katakni bajaring.')
log_text = LOG.read_text(encoding='utf-8')
if not any(tag in log_text for tag in ('QUALITY_STAGE_COMPLETE_CLOUD_VERIFIED', 'QUALITY_ALREADY_COMPLETE')):
    raise RuntimeError('Yakuniy bulut tasdig‘i hali yo‘q. Trening logini tekshiring; reset qilmang.')
model_lines = [line.removeprefix('MODEL:').strip() for line in log_text.splitlines() if line.startswith('MODEL:')]
if len(model_lines) != 1:
    raise RuntimeError('Model yo‘li noaniq; logni yuboring.')
MODEL_DIR = Path(model_lines[0])
if not MODEL_DIR.is_dir():
    raise RuntimeError('Runtime almashgan. Shu paket bilan 1–3 katakni qayta bajaring.')
sys.dont_write_bytecode = True
for folder in (KIT, KIT/'src'):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from stagequality.colab_runner import package_check, validate_result
_, _, BINDING, _ = package_check(KIT)
validate_result(MODEL_DIR.parent, BINDING)
comparison = json.loads((MODEL_DIR.parent/'comparison.json').read_text())
print('REAL-392 → 5K FULL MODEL; 60 Silver development, umumiy accuracy emas.')
for metric in ('exact_match_pct','raw_cer_pct','spelling_cer_pct','content_wer_pct',
               'punctuation_macro_f1_pct','apostrophe_f1_pct','casing_end_to_end_accuracy_pct',
               'identity_change_rate_pct'):
    old = comparison['before']['model']['apostrophe_equivalent'].get(metric)
    new = comparison['after']['model']['apostrophe_equivalent'].get(metric)
    print(metric, ':', old, '→', new)
print('Model:', MODEL_DIR)


## 5. O‘zingiz yozgan matn bilan sinov — ixtiyoriy
4-katak muvaffaqiyatli bajarilgandan keyin ishlating. Bu katak treningni davom ettirmaydi.
Modelning xom javobi; qo‘shimcha tuzatish qoidalari yo‘q. Baholash uchun ko‘rsatilgan matnlarni
keyin final test deb ishlatmang. Model GPU’da qoladi; boshqa treningni shu paytda boshlamang.


In [ ]:
if 'MODEL_DIR' not in globals():
    raise RuntimeError('Avval 4-katakni bajaring.')
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
if 'preview_model' not in globals() or globals().get('preview_path') != str(MODEL_DIR):
    preview_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, trust_remote_code=False)
    preview_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True,
                     use_safetensors=True, trust_remote_code=False, dtype=torch.float32).to('cuda').eval()
    preview_path = str(MODEL_DIR)
text = input('Matningizni kiriting: ')
if not text.strip() or len(text.encode('utf-8')) + 1 > 512:
    raise RuntimeError('Matn bo‘sh yoki 512 bayt chegarasidan uzun. Matn kesilmaydi.')
encoded = preview_tokenizer(text, return_tensors='pt').to('cuda')
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
    predicted = preview_model.generate(**encoded, max_length=513, num_beams=1, do_sample=False, use_cache=True)
print('INPUT :', text)
print('OUTPUT:', preview_tokenizer.decode(predicted[0], skip_special_tokens=True, clean_up_tokenization_spaces=False))
